In [ ]:
import os
import ast
import time
import gzip
import shutil
import logging
import requests
from datetime import datetime
import pandas as pd
import numpy as np
from kaggle.api.kaggle_api_extended import KaggleApi
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse.linalg import svds
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from argon2 import PasswordHasher, exceptions  # local import to avoid circular dependency issues
from pymongo import MongoClient, UpdateOne

class UserHandler:
    """Static helper methods for user creation, login, and initial rating collection."""
    
    @staticmethod
    def create_new_user(mongo_manager):
        latest_id = mongo_manager.get_latest_user_id()
        next_id = int(latest_id) + 1
        AppLogger.info(f"New user created with userId: {next_id}")
        password = input("Enter your password: ")
        from argon2 import PasswordHasher  # local import to avoid circular dependency issues
        ph = PasswordHasher()
        hashed_password = ph.hash(password)
        # Pass the mongo_manager to the new user so that future user logs include it.
        return User(userid=next_id, password=hashed_password, mongo_manager=mongo_manager)

    @staticmethod
    def load_existing_user(mongo_manager, user_id):
        existing_user = mongo_manager.get_user(user_id)
        if existing_user:
            input_password = input("Enter your password: ")
            from argon2 import PasswordHasher, exceptions
            ph = PasswordHasher()
            try:
                ph.verify(existing_user["password"], input_password)
                # Pass mongo_manager into the User object.
                user = User(
                    userid=existing_user["userId"],
                    password=existing_user["password"],
                    rated_movies=existing_user.get("rated_movies", []),
                    liked_movies=existing_user.get("liked_movies", []),
                    disliked_movies=existing_user.get("disliked_movies", []),
                    mongo_manager=mongo_manager
                )
                AppLogger.info("User loaded from MongoDB!", user, mongo_manager)
                return user
            except exceptions.VerifyMismatchError:
                AppLogger.error("Incorrect password. Exiting.", None, mongo_manager)
                return None
        else:
            AppLogger.error("User not found.", None, mongo_manager)
            return None

    @staticmethod
    def collect_initial_ratings(user, cleaned_movies_metadata, mongo_manager):
        AppLogger.info("Collecting initial ratings for new user.", user, mongo_manager)
        print("\n🎬 Please rate at least 3 movies to get started:")

        # Display a sample of popular movies as suggestions.
        popular_movies_pool = cleaned_movies_metadata.sort_values(by='popularity', ascending=False).head(50)[['id', 'title', 'genres']]
        current_sample = popular_movies_pool.sample(n=10)

        while True:
            print("\nSuggested movies to rate:")
            for _, movie in current_sample.iterrows():
                genres_str = ' | '.join(movie['genres']) if isinstance(movie['genres'], list) else movie['genres']
                print(f"{movie['id']}: {movie['title']} ({genres_str})")

            movie_id_input = input("\nEnter a movie ID to rate (or enter -1 to see a new selection): ")
            try:
                movie_id = int(movie_id_input)
            except ValueError:
                print("❌ Invalid input. Please enter a numeric movie ID or -1.")
                continue

            if movie_id == -1:
                current_sample = popular_movies_pool.sample(n=10)
                continue

            # Check if the movie_id exists in the entire dataset instead of only in the current sample.
            if movie_id not in cleaned_movies_metadata['id'].values:
                print("❌ Movie ID not found in the dataset. Please enter a valid movie ID.")
                continue

            try:
                rating = float(input("Rate this movie (0-5): "))
                if rating < 0 or rating > 5:
                    raise ValueError
            except ValueError:
                print("❌ Rating must be a numeric value between 0 and 5.")
                continue

            # Retrieve movie details from the full dataset.
            movie_details = cleaned_movies_metadata.loc[cleaned_movies_metadata['id'] == movie_id].iloc[0]
            movie_title = movie_details['title']
            movie_genres = movie_details['genres']

            user.add_rated_movie(movie_id, movie_title, rating, 'Initial', movie_genres)
            print(f"✅ Rated {movie_title}!")

            if len(user.rated_movies) >= 3:
                break

        print("\n🎉 Thank you! Your preferences have been saved.")


# ----------------------
# Logger and MongoDB Manager
# ----------------------
class AppLogger:
    """
    Updated logging system:
      - Maintains a logger for each source (typically a class name).
      - Each logger writes to both the console and a file in the "logs" directory.
      - Logging functions now require a 'source' parameter.
    """
    loggers = {}

    @staticmethod
    def get_logger(source="General"):
        if source not in AppLogger.loggers:
            logger = logging.getLogger(source)
            logger.setLevel(logging.INFO)
            if not logger.handlers:
                # Ensure the "logs" directory exists
                logs_directory = "logs"
                if not os.path.exists(logs_directory):
                    os.makedirs(logs_directory)
                
                # File handler writes to logs/<source>.txt
                file_path = os.path.join(logs_directory, f"{source}.txt")
                fh = logging.FileHandler(file_path)
                fh.setLevel(logging.INFO)
                formatter = logging.Formatter('[%(asctime)s] %(levelname)s: %(message)s')
                fh.setFormatter(formatter)
                logger.addHandler(fh)
                
                # Stream (console) handler
                ch = logging.StreamHandler()
                ch.setLevel(logging.INFO)
                ch.setFormatter(formatter)
                logger.addHandler(ch)
            AppLogger.loggers[source] = logger
        return AppLogger.loggers[source]

    @staticmethod
    def log_to_mongo(message, level, user, mongo_manager):
        log_entry = {
            "timestamp": datetime.now().isoformat(),
            "level": level,
            "message": message
        }
        try:
            mongo_manager.users_collection.update_one(
                {"userId": user.userid},
                {"$push": {"logs": log_entry}}
            )
        except Exception as e:
            AppLogger.get_logger("AppLogger").error(f"Error logging to MongoDB: {e}")

    @staticmethod
    def info(message, user=None, mongo_manager=None, source="General"):
        logger = AppLogger.get_logger(source)
        message_with_user = f"User {user.userid} - {message}" if user else message
        logger.info(message_with_user)
        if user and mongo_manager:
            AppLogger.log_to_mongo(message_with_user, "INFO", user, mongo_manager)

    @staticmethod
    def error(message, user=None, mongo_manager=None, source="General"):
        logger = AppLogger.get_logger(source)
        message_with_user = f"User {user.userid} - {message}" if user else message
        logger.error(message_with_user)
        if user and mongo_manager:
            AppLogger.log_to_mongo(message_with_user, "ERROR", user, mongo_manager)


class MongoDBManager:
    """
    A helper class to manage MongoDB read/write operations for users, movies, and ratings.
    Uses bulk operations and creates indexes to optimize performance.
    """
    def __init__(self, uri="mongodb://localhost:27017", db_name="movie_recommender"):
        self.client = MongoClient(uri)
        self.db = self.client[db_name]
        self.users_collection = self.db["users"]
        self.movies_collection = self.db["movies"]
        self.ratings_collection = self.db["ratings"]
        self._create_indexes()

    def _create_indexes(self):
        self.users_collection.create_index("userId", unique=True)
        self.movies_collection.create_index("id", unique=True)
        self.ratings_collection.create_index([("userId", 1), ("movieId", 1)], unique=True)

    def insert_or_update_user(self, user_data):
        try:
            self.users_collection.update_one(
                {"userId": user_data["userId"]},
                {"$set": user_data},
                upsert=True
            )
            AppLogger.info("User inserted/updated successfully.",
                           user=User(user_data["userId"], user_data["password"]),
                           mongo_manager=self,
                           source="MongoDBManager")
        except Exception as e:
            AppLogger.error(f"❌ Error inserting/updating user: {e}", source="MongoDBManager")

    def get_user(self, user_id):
        return self.users_collection.find_one({"userId": int(user_id)})

    def get_latest_user_id(self):
        user = self.users_collection.find_one(sort=[("userId", -1)])
        return user["userId"] if user else 0

    def insert_movies_bulk(self, movies_data):
        if not movies_data:
            return
        operations = [
            UpdateOne({"id": movie["id"]}, {"$set": movie}, upsert=True)
            for movie in movies_data
        ]
        try:
            self.movies_collection.bulk_write(operations)
            AppLogger.info("Movies metadata bulk write completed.", source="MongoDBManager")
        except Exception as e:
            AppLogger.error(f"❌ Error during bulk writing movies: {e}", source="MongoDBManager")

    def get_movie(self, movie_id):
        return self.movies_collection.find_one({"id": movie_id})

    def get_all_movies(self, query={}):
        return list(self.movies_collection.find(query))

    def insert_ratings_bulk(self, ratings_data):
        if not ratings_data:
            return
        operations = [
            UpdateOne(
                {"userId": rating["userId"], "movieId": rating["movieId"]},
                {"$set": rating},
                upsert=True
            )
            for rating in ratings_data
        ]
        try:
            self.ratings_collection.bulk_write(operations)
            AppLogger.info("Ratings bulk write completed.", source="MongoDBManager")
        except Exception as e:
            AppLogger.error(f"❌ Error during bulk writing ratings: {e}", source="MongoDBManager")

    def get_ratings(self, query={}):
        return list(self.ratings_collection.find(query))

# ----------------------
# Updated Data Ingestion Class
# ----------------------
class DataIngestion:
    def __init__(self, dataset_path, download_path='./dataset', 
                 imdb_crew_url="https://datasets.imdbws.com/title.crew.tsv.gz",
                 imdb_names_url="https://datasets.imdbws.com/name.basics.tsv.gz"):
        """
        Parameters:
            dataset_path: Kaggle dataset identifier (e.g., "shubhammehta21/movie-lens-small-latest-dataset")
            download_path: Local directory where datasets will be saved
            imdb_crew_url: URL for the IMDb title.crew.tsv.gz file
            imdb_names_url: URL for the IMDb name.basics.tsv.gz file
        """
        self.dataset_path = dataset_path
        self.download_path = download_path
        self.imdb_crew_url = imdb_crew_url
        self.imdb_names_url = imdb_names_url
        self.imdb_crew_path = None
        self.imdb_names_path = None
        
        os.makedirs(self.download_path, exist_ok=True)
        
        self.api = KaggleApi()
        try:
            self.api.authenticate()
        except Exception as e:
            raise Exception("Kaggle API authentication failed.") from e

    @staticmethod
    def get_director_names(df_crew: pd.DataFrame, df_names: pd.DataFrame) -> pd.DataFrame:
        df_crew = df_crew[df_crew['directors'] != '\\N'].copy()
        df_crew['director'] = df_crew['directors'].str.split(',')
        df_exploded = df_crew.explode('director')
        merged_df = df_exploded.merge(
            df_names[['nconst', 'primaryName']],
            left_on='director',
            right_on='nconst',
            how='left'
        )
        result_df = merged_df[['tconst', 'primaryName']].rename(
            columns={'tconst': 'imdbid', 'primaryName': 'directorname'}
        )
        return result_df

    def pad_imdb_id(self, imdb_id) -> str:
        try:
            imdb_id_int = int(imdb_id)
            return f"tt{imdb_id_int:07d}"
        except ValueError:
            return None

    def download_data(self, download_path=None):
        if download_path is None:
            download_path = self.download_path
        start_time = time.time()
        self.api.dataset_download_files(self.dataset_path, path=download_path, unzip=True)
        elapsed = time.time() - start_time
        AppLogger.info(f"Downloaded MovieLens dataset in {elapsed:.2f} seconds.", source="DataIngestion")
        AppLogger.info("Listing contents of MovieLens directory:", source="DataIngestion")
        for root, _, files in os.walk(download_path):
            for file in files:
                AppLogger.info(f" - {file}", source="DataIngestion")
        required_files = ['movies.csv', 'ratings.csv', 'links.csv']
        missing_files = [f for f in required_files if not os.path.exists(os.path.join(download_path, f))]
        if missing_files:
            AppLogger.error(f"Missing files: {missing_files}", source="DataIngestion")
            return None
        try:
            movies = pd.read_csv(os.path.join(download_path, 'movies.csv'))
            ratings = pd.read_csv(os.path.join(download_path, 'ratings.csv'))
            links = pd.read_csv(os.path.join(download_path, 'links.csv'))
            return movies, ratings, links
        except Exception as e:
            AppLogger.error(f"Error loading MovieLens data: {e}", source="DataIngestion")
            return None

    def download_imdb_data(self):
        imdb_files = {
            "title.crew.tsv": self.imdb_crew_url,
            "name.basics.tsv": self.imdb_names_url
        }
        for filename, url in imdb_files.items():
            local_gz_path = os.path.join(self.download_path, filename + ".gz")
            local_tsv_path = os.path.join(self.download_path, filename)
            if not os.path.exists(local_tsv_path):
                AppLogger.info(f"Downloading {filename} from {url} ...", source="DataIngestion")
                try:
                    response = requests.get(url, stream=True)
                    response.raise_for_status()
                    with open(local_gz_path, 'wb') as f:
                        f.write(response.content)
                    with gzip.open(local_gz_path, 'rb') as f_in:
                        with open(local_tsv_path, 'wb') as f_out:
                            shutil.copyfileobj(f_in, f_out)
                    AppLogger.info(f"Saved {local_tsv_path}", source="DataIngestion")
                except Exception as e:
                    AppLogger.error(f"Error downloading {filename}: {e}", source="DataIngestion")
            else:
                AppLogger.info(f"{filename} already exists locally.", source="DataIngestion")
            if filename == "title.crew.tsv":
                self.imdb_crew_path = local_tsv_path
            elif filename == "name.basics.tsv":
                self.imdb_names_path = local_tsv_path

    def load_data(self, download_path=None):
        if download_path is None:
            download_path = self.download_path
        data_files = {}
        for file in os.listdir(download_path):
            if file.endswith(".csv"):
                file_path = os.path.join(download_path, file)
                df_name = file.replace(".csv", "")
                data_files[df_name] = pd.read_csv(file_path)
                AppLogger.info(f"Loaded {file}", source="DataIngestion")
        return data_files

    def download_and_load(self, download_path=None):
        if download_path is None:
            download_path = self.download_path
        self.download_data(download_path)
        return self.load_data(download_path)

    def ingest_movie_data(self):
        data = self.download_data(self.download_path)
        if data is None:
            raise Exception("MovieLens dataset download failed or required files are missing.")
        movies, ratings, links = data
        links['imdbid'] = links['imdbId'].apply(self.pad_imdb_id)
        movies_merged = pd.merge(movies, links[['movieId', 'imdbid']], on='movieId', how='left')
        self.download_imdb_data()
        if self.imdb_crew_path and self.imdb_names_path:
            try:
                df_crew = pd.read_csv(self.imdb_crew_path, sep='\t', dtype=str)
                df_names = pd.read_csv(self.imdb_names_path, sep='\t', dtype=str)
                directors_df = self.get_director_names(df_crew, df_names)
                movies_merged = pd.merge(movies_merged, directors_df, on='imdbid', how='left')
            except Exception as e:
                AppLogger.error(f"Error loading IMDb director data: {e}", source="DataIngestion")
        else:
            AppLogger.info("IMDb crew or names data not available. Skipping director merge.", source="DataIngestion")
        return movies_merged

# ----------------------
# Movie Preprocessor (unchanged except for working with movies from ingestion)
# ----------------------
class MoviePreprocessor:
    def __init__(self, movies_metadata, ratings):
        self.movies_metadata = movies_metadata.copy()
        self.ratings = ratings

    def clean_movies_metadata(self):
        AppLogger.info("Cleaning movies metadata...", source="MoviePreprocessor")
        start_time = time.time()
        self.movies_metadata = self.movies_metadata.dropna(subset=['movieId', 'title'])
        self.movies_metadata['movieId'] = pd.to_numeric(self.movies_metadata['movieId'], errors='coerce')
        self.movies_metadata = self.movies_metadata.dropna(subset=['movieId'])
        self.movies_metadata['movieId'] = self.movies_metadata['movieId'].astype(int)
        self.movies_metadata = self.movies_metadata.rename(columns={'movieId': 'id', 'directorname': 'director'})
        self.movies_metadata['genres'] = self.movies_metadata['genres'].fillna('')
        self.movies_metadata['genres'] = self.movies_metadata['genres'].apply(
            lambda x: x.split('|') if isinstance(x, str) and x != '(no genres listed)' else []
        )
        popularity = self.ratings['movieId'].value_counts().reset_index()
        popularity.columns = ['id', 'popularity']
        self.movies_metadata = pd.merge(self.movies_metadata, popularity, on='id', how='left')
        self.movies_metadata['popularity'] = self.movies_metadata['popularity'].fillna(0)
        elapsed = time.time() - start_time
        AppLogger.info(f"Movies metadata cleaned in {elapsed:.2f} seconds!", source="MoviePreprocessor")

    def clean_ratings(self):
        AppLogger.info("Cleaning ratings data...", source="MoviePreprocessor")
        start_time = time.time()
        self.ratings = self.ratings.dropna(subset=['userId', 'movieId', 'rating'])
        self.ratings['userId'] = pd.to_numeric(self.ratings['userId'], errors='coerce')
        self.ratings['movieId'] = pd.to_numeric(self.ratings['movieId'], errors='coerce')
        self.ratings = self.ratings.dropna(subset=['userId', 'movieId'])
        self.ratings['rating'] = self.ratings['rating'].fillna(0)
        valid_movie_ids = self.movies_metadata['id'].unique()
        self.ratings = self.ratings[self.ratings['movieId'].isin(valid_movie_ids)]
        self.ratings = self.ratings.drop_duplicates(subset=['userId', 'movieId'])
        elapsed = time.time() - start_time
        AppLogger.info(f"Ratings cleaned in {elapsed:.2f} seconds!", source="MoviePreprocessor")

    def preprocess_all(self):
        self.clean_movies_metadata()
        self.clean_ratings()
        return {
            "movies_metadata": self.movies_metadata[['id', 'title', 'genres', 'popularity', 'director']],
            "ratings": self.ratings
        }
    
    def store_to_db(self, mongo_manager):
        AppLogger.info("Storing data to MongoDB...", source="MoviePreprocessor")
        start_time = time.time()
        movies_data = self.movies_metadata[['id', 'title', 'genres', 'popularity', 'director']].to_dict("records")
        for movie in movies_data:
            movie["id"] = int(movie["id"])
            movie["popularity"] = float(movie["popularity"])
        mongo_manager.insert_movies_bulk(movies_data)
        
        ratings_data = self.ratings.to_dict("records")
        for rating in ratings_data:
            rating["userId"] = int(rating["userId"])
            rating["movieId"] = int(rating["movieId"])
            rating["rating"] = float(rating["rating"])
        mongo_manager.insert_ratings_bulk(ratings_data)
        
        user_ids = self.ratings['userId'].unique()
        for uid in user_ids:
            uid_int = int(uid)
            user_ratings = self.ratings[self.ratings['userId'] == uid]
            rated_movies = user_ratings.to_dict('records')
            liked_movies = user_ratings[user_ratings['rating'] >= 3].to_dict('records')
            disliked_movies = user_ratings[user_ratings['rating'] < 3].to_dict('records')
            for record in rated_movies:
                record["userId"] = int(record["userId"])
                record["movieId"] = int(record["movieId"])
                record["rating"] = float(record["rating"])
            for record in liked_movies:
                record["userId"] = int(record["userId"])
                record["movieId"] = int(record["movieId"])
                record["rating"] = float(record["rating"])
            for record in disliked_movies:
                record["userId"] = int(record["userId"])
                record["movieId"] = int(record["movieId"])
                record["rating"] = float(record["rating"])
            existing_user = mongo_manager.get_user(uid_int)
            if not existing_user:
                ph = PasswordHasher()
                hashed_pass = ph.hash(str(uid_int))
                user_doc = {
                    "userId": uid_int,
                    "password": hashed_pass,
                    "rated_movies": rated_movies,
                    "liked_movies": liked_movies,
                    "disliked_movies": disliked_movies
                }
                mongo_manager.insert_or_update_user(user_doc)
        elapsed = time.time() - start_time
        AppLogger.info(f"Movies metadata, ratings, and existing users stored in MongoDB in {elapsed:.2f} seconds.",
                       source="MoviePreprocessor")

# ----------------------
# Recommender System Classes (unchanged except for logging sources)
# ----------------------
class SVDBasedRecommender:
    def __init__(self, ratings, movies_metadata):
        self.ratings = ratings
        self.movies_metadata = movies_metadata
        self.model = None
        self.predicted_ratings = None
        self.user_means = None
        self.user_movie_ratings = None
        self.movie_ids = []

    def prepare_user_item_matrix(self):
        df = self.ratings.groupby(['userId', 'movieId'], as_index=False).agg({'rating': 'mean'})
        self.user_movie_ratings = df.pivot(index='userId', columns='movieId', values='rating').fillna(0)
        self.user_means = self.user_movie_ratings.mean(axis=1)
        return self.user_movie_ratings

    def train_svd(self, num_features=50):
        AppLogger.info("Training SVD model...", source="SVDBasedRecommender")
        start_time = time.time()
        user_movie_ratings = self.prepare_user_item_matrix()
        self.movie_ids = user_movie_ratings.columns.tolist()
        normalized_ratings = user_movie_ratings.sub(self.user_means, axis=0)
        sparse_ratings = csr_matrix(normalized_ratings.values)
        U, sigma, Vt = svds(sparse_ratings, k=min(num_features, sparse_ratings.shape[1] - 1))
        sigma = np.diag(sigma)
        self.U, self.sigma, self.Vt = U, sigma, Vt
        self.predict_ratings()
        elapsed = time.time() - start_time
        AppLogger.info(f"SVD training completed in {elapsed:.2f} seconds!", source="SVDBasedRecommender")

    def predict_ratings(self):
        if not hasattr(self, 'U') or not hasattr(self, 'sigma') or not hasattr(self, 'Vt'):
            raise ValueError("❌ SVD model is not trained. Run `train_svd()` first.")
        self.predicted_ratings = np.dot(np.dot(self.U, self.sigma), self.Vt) + self.user_means.values[:, np.newaxis]

    def recommend_movies(self, user, n_recommendations=5, filter_rated=True):
        if self.predicted_ratings is None:
            raise ValueError("❌ Model not trained. Run `train_svd()` first.")
        if self.user_movie_ratings is None or user.userid not in self.user_movie_ratings.index:
            raise IndexError(f"❌ User ID {user.userid} not found in the ratings matrix. Please retrain the model with the new user data.")
        user_idx = self.user_movie_ratings.index.get_loc(user.userid)
        predicted_scores = self.predicted_ratings[user_idx]
        recommendations = pd.DataFrame({'id': self.movie_ids, 'predicted_rating': predicted_scores})
        recommendations = recommendations.merge(self.movies_metadata[['id', 'title', 'genres']], on='id', how='left')
        if filter_rated:
            user_rated = set(user.rated_movies['movieId'].values)
            recommendations = recommendations[~recommendations['id'].isin(user_rated)]
        recommendations = recommendations.sort_values(by='predicted_rating', ascending=False)
        recs = recommendations.head(n_recommendations).copy()
        recs['modelName'] = 'svd'
        return recs

class ContentBasedRecommender:
    def __init__(self, movies_metadata):
        self.movies_metadata = movies_metadata.copy()
        self.tfidf_matrix = None
        self.cosine_sim = None
        self.movie_id_to_idx = None
        self.prepare_content_features()

    def prepare_content_features(self):
        AppLogger.info("Preparing content features...", source="ContentBasedRecommender")
        start_time = time.time()
        self.movies_metadata['genres'] = self.movies_metadata['genres'].fillna('')
        self.movies_metadata['content'] = self.movies_metadata['genres'].apply(lambda x: ' '.join(x))
        tfidf = TfidfVectorizer(stop_words='english', max_features=5000, sublinear_tf=True)
        self.tfidf_matrix = tfidf.fit_transform(self.movies_metadata['content'])
        self.cosine_sim = cosine_similarity(self.tfidf_matrix, dense_output=False)
        self.movie_id_to_idx = pd.Series(self.movies_metadata.index, index=self.movies_metadata['id']).drop_duplicates()
        elapsed = time.time() - start_time
        AppLogger.info(f"Content features prepared in {elapsed:.2f} seconds!", source="ContentBasedRecommender")

    def recommend_movies(self, user, n_recommendations=5, filter_rated=True):
        rated_movies = user.rated_movies[user.rated_movies['rating'] >= 2.5]
        if rated_movies.empty:
            AppLogger.info("No liked movies found. Returning popular movies as fallback.", user, user.mongo_manager, source="ContentBasedRecommender")
            fallback_recs = self.movies_metadata.sort_values(by='popularity', ascending=False).head(n_recommendations).copy()
            fallback_recs['similarity_score'] = np.nan
            fallback_recs['modelName'] = 'content-based'
            return fallback_recs
        liked_ids = rated_movies['movieId'].values
        liked_indices = self.movie_id_to_idx[liked_ids].dropna().astype(int).values
        sim_submatrix = self.cosine_sim[:, liked_indices]
        avg_sim_scores = np.asarray(sim_submatrix.mean(axis=1)).flatten()
        rec_df = self.movies_metadata.copy()
        rec_df['avg_sim_score'] = avg_sim_scores
        if filter_rated:
            rated_set = set(user.rated_movies['movieId'].values)
            rec_df = rec_df[~rec_df['id'].isin(rated_set)]
        rec_df = rec_df.sort_values('avg_sim_score', ascending=False)
        recs = rec_df.head(n_recommendations).copy()
        recs['modelName'] = 'content-based'
        return recs

    def retrain(self):
        self.prepare_content_features()
        AppLogger.info("Content-Based Model retrained successfully!", source="ContentBasedRecommender")

class HybridRecommender:
    def __init__(self, svd_model, content_model, user):
        self.svd_model = svd_model
        self.content_model = content_model
        self.user = user

    def recommend_movies(self, user=None, n_recommendations=5, filter_rated=True):
        svd_recs = self.svd_model.recommend_movies(self.user, n_recommendations, filter_rated=filter_rated)
        content_recs = self.content_model.recommend_movies(self.user, n_recommendations, filter_rated=filter_rated)
        svd_recs = svd_recs[["id", "title", "genres", "predicted_rating"]].rename(
            columns={"predicted_rating": "score"}
        )
        content_recs = content_recs[["id", "title", "genres", "avg_sim_score"]].rename(
            columns={"avg_sim_score": "score"}
        )
        combined = pd.concat([svd_recs, content_recs]).drop_duplicates(subset=["id"])
        hybrid_recs = combined.sample(n=n_recommendations, random_state=42)
        hybrid_recs['modelName'] = 'hybrid'
        return hybrid_recs

    def retrain_models(self, feedback_df):
        AppLogger.info("Retraining Hybrid model based on feedback...", source="HybridRecommender")
        self.svd_model.ratings = pd.concat([self.svd_model.ratings, feedback_df[['userId', 'movieId', 'rating']]])
        self.svd_model.prepare_user_item_matrix()
        self.svd_model.train_svd()
        self.content_model.prepare_content_features()

# ----------------------
# User and Feedback Classes
# ----------------------
class User:
    def __init__(self, userid, password, roles=None, rated_movies=None, liked_movies=None, disliked_movies=None, mongo_manager=None):
        self.userid = int(userid)
        self.password = password  # This should be a hashed password!
        self.roles = roles if roles else []
        self.mongo_manager = mongo_manager
        self.rated_movies = (pd.DataFrame(rated_movies) if rated_movies is not None 
                             else pd.DataFrame(columns=["userId", "movieId", "title", "rating", "modelName", "genre_names"]))
        self.liked_movies = (pd.DataFrame(liked_movies) if liked_movies is not None 
                             else pd.DataFrame(columns=["userId", "movieId", "title", "rating", "modelName", "genre_names"]))
        self.disliked_movies = (pd.DataFrame(disliked_movies) if disliked_movies is not None 
                             else pd.DataFrame(columns=["userId", "movieId", "title", "rating", "modelName", "genre_names"]))

    def register(self, userid, password):
        ph = PasswordHasher()
        self.userid = userid
        self.password = ph.hash(password)
        AppLogger.info(f"User {self.userid} is registered.", self, self.mongo_manager, source="User")

    def login(self, input_password):
        ph = PasswordHasher()
        try:
            ph.verify(self.password, input_password)
            AppLogger.info("Login successful.", self, self.mongo_manager, source="User")
        except exceptions.VerifyMismatchError:
            AppLogger.error("Incorrect password.", self, self.mongo_manager, source="User")

    def change_password(self, new_password):
        ph = PasswordHasher()
        self.password = ph.hash(new_password)
        AppLogger.info("Password changed successfully.", self, self.mongo_manager, source="User")

    def delete_movie_history(self):
        self.rated_movies = pd.DataFrame(columns=["userId", "movieId", "title", "rating", "modelName", "genre_names"])
        self.liked_movies = pd.DataFrame(columns=["userId", "movieId", "title", "rating", "modelName", "genre_names"])
        self.disliked_movies = pd.DataFrame(columns=["userId", "movieId", "title", "rating", "modelName", "genre_names"])
        AppLogger.info("Movie history deleted.", self, self.mongo_manager, source="User")

    def get_initial_movies(self, initial_movies):
        self.rated_movies = pd.concat([self.rated_movies, initial_movies], ignore_index=True)
        AppLogger.info("Initial movies rated and saved.", self, self.mongo_manager, source="User")

    def add_rated_movie(self, movie_id, title, rating, model_name, genre_names):
        movie_id = int(movie_id)
        new_entry = {
            "userId": self.userid,
            "movieId": movie_id,
            "title": title,
            "rating": rating,
            "modelName": model_name,
            "genre_names": genre_names
        }
        new_row = pd.DataFrame([new_entry])
        self.rated_movies = pd.concat([self.rated_movies, new_row], ignore_index=True)
        if rating >= 3:
            self.liked_movies = pd.concat([self.liked_movies, new_row], ignore_index=True)
        else:
            self.disliked_movies = pd.concat([self.disliked_movies, new_row], ignore_index=True)
        AppLogger.info(f"Movie '{title}' rated and saved.", self, self.mongo_manager, source="User")

    def to_dict(self):
        return {
            "userId": self.userid,
            "password": self.password,
            "rated_movies": self.rated_movies.to_dict("records"),
            "liked_movies": self.liked_movies.to_dict("records"),
            "disliked_movies": self.disliked_movies.to_dict("records")
        }

class Feedback:
    def __init__(self, movies, user, svd_model, content_model, hybrid_model, mongo_manager):
        self.movies = movies
        self.user = user
        self.svd_model = svd_model
        self.content_model = content_model
        self.hybrid_model = hybrid_model
        self.mongo_manager = mongo_manager
        self.relevant_movies = set()
        self.update_relevant_movies()

    def update_relevant_movies(self):
        self.relevant_movies = set(self.user.rated_movies[self.user.rated_movies['rating'] >= 3]['movieId'])

    def calculate_mapk(self, recommendations, k=5):
        recommended_ids = recommendations['id'].tolist()[:k]
        if not self.relevant_movies:
            AppLogger.info("No relevant movies found for this user.", self.user, self.mongo_manager, source="Feedback")
            return 0.0  
        relevant_in_top_k = sum(1 for mid in recommended_ids if mid in self.relevant_movies)
        return relevant_in_top_k / k

    def calculate_recommendation_accuracy(self, recommendations):
        liked_set = set(self.user.liked_movies['movieId'])
        recommended_ids = recommendations['id'].tolist()
        if not recommended_ids:
            return 0.0
        correct = sum(1 for mid in recommended_ids if mid in liked_set)
        return correct / len(recommended_ids)
    
    def calculate_model_accuracies(self):
        results = {}
        if "modelName" not in self.user.rated_movies.columns:
            self.user.rated_movies["modelName"] = ""
        for model in ['svd', 'content-based', 'hybrid']:
            model_ratings = self.user.rated_movies[self.user.rated_movies['modelName'].str.lower() == model]
            total = len(model_ratings)
            if total == 0:
                accuracy = 0.0
            else:
                liked = model_ratings[model_ratings['rating'] >= 3]
                accuracy = len(liked) / total
            results[model] = accuracy
        return results

    def evaluate_models(self, k=5):
        results = {}
        svd_recs = self.svd_model.recommend_movies(self.user, n_recommendations=k, filter_rated=False)
        results['svd'] = self.calculate_mapk(svd_recs, k=k)
        content_recs = self.content_model.recommend_movies(self.user, n_recommendations=k, filter_rated=False)
        results['content-based'] = self.calculate_mapk(content_recs, k=k)
        hybrid_recs = self.hybrid_model.recommend_movies(self.user, n_recommendations=k, filter_rated=False)
        results['hybrid'] = self.calculate_mapk(hybrid_recs, k=k)
        return results

    def feedback_loop(self, active_model):
        self.update_relevant_movies()
        while True:
            recommendations = active_model.recommend_movies(self.user, 5)
            print("\nRecommended Movies:")
            print(recommendations[['id', 'title', 'genres', 'modelName']])
            
            model_results = self.evaluate_models(k=5)
            print("\n📊 Model Precision@5 Scores:")
            for model, score in model_results.items():
                print(f"- {model.capitalize()}: {score:.4f}")
            
            model_accs = self.calculate_model_accuracies()
            print("\n🔍 Model Accuracy (liked/rated) for the current user:")
            for model_key, acc in model_accs.items():
                print(f"- {model_key.capitalize()} ACC: {acc:.4f}")
            
            user_feedback = self.collect_feedback(recommendations)
            if user_feedback is not None:
                self.mongo_manager.insert_ratings_bulk(user_feedback.to_dict("records"))
                self.mongo_manager.insert_or_update_user(self.user.to_dict())
                self.adjust_model_based_on_feedback(active_model, user_feedback)
                self.update_relevant_movies()

            cont = input("Continue feedback loop? (yes/no): ").lower()
            if cont != 'yes':
                break

    def collect_feedback(self, recommendations):
        feedback_data = []
        for _, row in recommendations.iterrows():
            while True:
                try:
                    feedback = float(input(f"Rate '{row['title']}' (0-5): "))
                    if 0 <= feedback <= 5:
                        break
                    else:
                        print("❌ Rating must be between 0 and 5.")
                except ValueError:
                    print("❌ Invalid input. Please enter a numeric value between 0 and 5.")
            feedback_data.append({
                "userId": self.user.userid,
                "movieId": row['id'],
                "title": row['title'],
                "rating": feedback,
                "modelName": row.get("modelName", "feedback"),
                "genre_names": ', '.join(row['genres']) if isinstance(row['genres'], list) else row['genres']
            })
        if feedback_data:
            feedback_df = pd.DataFrame(feedback_data)
            for entry in feedback_data:
                self.user.add_rated_movie(entry["movieId"], entry["title"], entry["rating"], entry["modelName"], entry["genre_names"])
            return feedback_df
        return None

    def adjust_model_based_on_feedback(self, model, user_feedback):
        AppLogger.info("Retraining the model with user feedback...", self.user, self.mongo_manager, source="Feedback")
        if isinstance(model, HybridRecommender):
            model.svd_model.ratings = pd.concat([model.svd_model.ratings, user_feedback[['userId', 'movieId', 'rating']]])
            model.svd_model.prepare_user_item_matrix()
            model.svd_model.train_svd()
            model.content_model.prepare_content_features()
        elif isinstance(model, SVDBasedRecommender):
            model.ratings = pd.concat([model.ratings, user_feedback[['userId', 'movieId', 'rating']]])
            model.prepare_user_item_matrix()
            model.train_svd()
        elif isinstance(model, ContentBasedRecommender):
            AppLogger.info("Updating content-based model with new feedback...", self.user, self.mongo_manager, source="Feedback")
            model.prepare_content_features()
            AppLogger.info("Content-based model updated!", self.user, self.mongo_manager, source="Feedback")
        AppLogger.info("Model retrained successfully.", self.user, self.mongo_manager, source="Feedback")

def calculate_apk(recommended_ids, relevant_ids, k=5):
    ap = 0.0
    num_hits = 0
    for i, movie_id in enumerate(recommended_ids[:k], 1):
        if movie_id in relevant_ids:
            num_hits += 1
            precision_at_i = num_hits / i
            ap += precision_at_i
    num_relevant = min(k, len(relevant_ids))
    if num_relevant == 0:
        return 0.0
    return ap / num_relevant

class MLPipeline:
    def __init__(self, dataset_url):
        self.dataset_url = dataset_url
        self.data_ingestion = DataIngestion(dataset_url)
        self.model_name = ""
        self.user = None
        self.active_model = None
        self.svd_model = None
        self.content_model = None
        self.hybrid_model = None
        self.cleaned_movies_metadata = None
        self.cleaned_ratings = None
        self.mongo_manager = MongoDBManager()

    def build_pipeline(self):
        AppLogger.info("Starting pipeline...", source="MLPipeline")
        start_pipeline = time.time()
        download_path = self.data_ingestion.download_path
        downloaded_files = ['movies.csv', 'ratings.csv', 'links.csv']
        files_exist = all(os.path.exists(os.path.join(download_path, f)) for f in downloaded_files)
        mongo_movies_count = self.mongo_manager.movies_collection.count_documents({})
        
        if files_exist and mongo_movies_count > 0:
            AppLogger.info("Data already downloaded, preprocessed, and stored in MongoDB. Loading from database...", source="MLPipeline")
            movies = list(self.mongo_manager.movies_collection.find())
            ratings = list(self.mongo_manager.ratings_collection.find())
            self.cleaned_movies_metadata = pd.DataFrame(movies)
            self.cleaned_ratings = pd.DataFrame(ratings)
            # Drop MongoDB generated _id field if it exists
            if '_id' in self.cleaned_movies_metadata.columns:
                self.cleaned_movies_metadata.drop(columns=['_id'], inplace=True)
            if '_id' in self.cleaned_ratings.columns:
                self.cleaned_ratings.drop(columns=['_id'], inplace=True)
        else:
            AppLogger.info("Data not found in MongoDB. Proceeding with data ingestion, preprocessing, and storing...", source="MLPipeline")
            movies_metadata_enriched = self.data_ingestion.ingest_movie_data()
            data = self.data_ingestion.download_data()
            if data is None:
                raise ValueError("Failed to load data. Exiting pipeline.")
            _, ratings, _ = data
            AppLogger.info("Data ingested successfully!", source="MLPipeline")

            AppLogger.info("Preprocessing data...", source="MLPipeline")
            start_preproc = time.time()
            preprocessor = MoviePreprocessor(movies_metadata_enriched, ratings)
            processed_data = preprocessor.preprocess_all()
            self.cleaned_movies_metadata = processed_data["movies_metadata"]
            self.cleaned_ratings = processed_data["ratings"]
            elapsed_preproc = time.time() - start_preproc
            AppLogger.info(f"Data preprocessed in {elapsed_preproc:.2f} seconds!", source="MLPipeline")

            AppLogger.info("Storing movies metadata, ratings, and existing users to MongoDB via preprocessor...", source="MLPipeline")
            preprocessor.store_to_db(self.mongo_manager)

        AppLogger.info("Setting up user...", source="MLPipeline")
        try:
            userid = int(input("Enter your user ID (or 0 to create new user): "))
        except ValueError:
            AppLogger.error("Invalid input. Exiting.", source="MLPipeline")
            return

        is_new_user = False
        if userid != 0:
            user = UserHandler.load_existing_user(self.mongo_manager, userid)
            if user is None:
                return
            self.user = user
        else:
            self.user = UserHandler.create_new_user(self.mongo_manager)
            UserHandler.collect_initial_ratings(self.user, self.cleaned_movies_metadata, self.mongo_manager)
            is_new_user = True

        self.mongo_manager.insert_or_update_user(self.user.to_dict())
        AppLogger.info("User setup complete.", self.user, self.mongo_manager, source="MLPipeline")

        AppLogger.info("Initializing models...", source="MLPipeline")
        self.svd_model = SVDBasedRecommender(self.cleaned_ratings, self.cleaned_movies_metadata)
        self.content_model = ContentBasedRecommender(self.cleaned_movies_metadata)

        if is_new_user:
            AppLogger.info(f"New user detected (User ID: {self.user.userid}). Retraining models with initial ratings...", self.user, self.mongo_manager, source="MLPipeline")
            self.cleaned_ratings = pd.concat([self.cleaned_ratings, self.user.rated_movies], ignore_index=True)
            self.svd_model.ratings = self.cleaned_ratings
            self.svd_model.train_svd()
            self.content_model.retrain()
            new_ratings_list = self.user.rated_movies.to_dict("records")
            self.mongo_manager.insert_ratings_bulk(new_ratings_list)
        else:
            self.svd_model.train_svd()
            self.content_model.retrain()

        self.hybrid_model = HybridRecommender(self.svd_model, self.content_model, self.user)
        AppLogger.info("Models initialized!", self.user, self.mongo_manager, source="MLPipeline")

        self.model_name = input("Choose a model (SVD, Content-Based, Hybrid): ").strip().lower()
        if self.model_name == "svd":
            self.active_model = self.svd_model
        elif self.model_name == "content-based":
            self.active_model = self.content_model
        elif self.model_name == "hybrid":
            self.active_model = self.hybrid_model
        else:
            raise ValueError("Invalid model choice.")

        feedback = Feedback(self.cleaned_movies_metadata, self.user, self.svd_model, self.content_model, self.hybrid_model, self.mongo_manager)
        feedback.feedback_loop(self.active_model)
        elapsed_pipeline = time.time() - start_pipeline
        AppLogger.info(f"Pipeline completed in {elapsed_pipeline:.2f} seconds.", self.user, self.mongo_manager, source="MLPipeline")

def main():
    dataset_url = "shubhammehta21/movie-lens-small-latest-dataset"
    pipeline = MLPipeline(dataset_url)
    pipeline.build_pipeline()

if __name__ == "__main__":
    main()


[2025-03-04 15:57:55,718] INFO: Starting pipeline...
[2025-03-04 15:57:55,738] INFO: Data already downloaded, preprocessed, and stored in MongoDB. Loading from database...
[2025-03-04 15:57:56,654] INFO: Setting up user...


Enter your user ID (or 0 to create new user): 0


[2025-03-04 15:57:58,200] INFO: New user created with userId: 614


Enter your password: 0


[2025-03-04 15:58:00,093] INFO: User 614 - Collecting initial ratings for new user.


...

✅ Rated Fair Game (1995)!

Suggested movies to rate:
1196: Star Wars: Episode V - The Empire Strikes Back (1980) (Action | Adventure | Sci-Fi)
1270: Back to the Future (1985) (Adventure | Comedy | Sci-Fi)
592: Batman (1989) (Action | Crime | Thriller)
4993: Lord of the Rings: The Fellowship of the Ring, The (2001) (Adventure | Fantasy)
296: Pulp Fiction (1994) (Comedy | Crime | Drama | Thriller)
380: True Lies (1994) (Action | Adventure | Comedy | Romance | Thriller)
2858: American Beauty (1999) (Drama | Romance)
1198: Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981) (Action | Adventure)
283995: Guardians of the Galaxy Vol. 2 (Action | Adventure | Comedy | Science Fiction)
780: Independence Day (a.k.a. ID4) (1996) (Action | Adventure | Sci-Fi | Thriller)

Enter a movie ID to rate (or enter -1 to see a new selection): 5456
Rate this movie (0-5): 5


[2025-03-04 15:58:24,883] INFO: User 614 - Movie 'Wagons East (1994)' rated and saved.


✅ Rated Wagons East (1994)!

Suggested movies to rate:
1196: Star Wars: Episode V - The Empire Strikes Back (1980) (Action | Adventure | Sci-Fi)
1270: Back to the Future (1985) (Adventure | Comedy | Sci-Fi)
592: Batman (1989) (Action | Crime | Thriller)
4993: Lord of the Rings: The Fellowship of the Ring, The (2001) (Adventure | Fantasy)
296: Pulp Fiction (1994) (Comedy | Crime | Drama | Thriller)
380: True Lies (1994) (Action | Adventure | Comedy | Romance | Thriller)
2858: American Beauty (1999) (Drama | Romance)
1198: Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981) (Action | Adventure)
283995: Guardians of the Galaxy Vol. 2 (Action | Adventure | Comedy | Science Fiction)
780: Independence Day (a.k.a. ID4) (1996) (Action | Adventure | Sci-Fi | Thriller)

Enter a movie ID to rate (or enter -1 to see a new selection): 7021
❌ Movie ID not found in the dataset. Please enter a valid movie ID.

Suggested movies to rate:
1196: Star Wars: Episode V - The Empire

[2025-03-04 15:58:45,634] INFO: User 614 - Movie 'Star Wars: Episode V - The Empire Strikes Back (1980)' rated and saved.
[2025-03-04 15:58:45,673] INFO: User 614 - User inserted/updated successfully.
[2025-03-04 15:58:45,678] INFO: User 614 - User setup complete.
[2025-03-04 15:58:45,680] INFO: Initializing models...
[2025-03-04 15:58:45,686] INFO: Preparing content features...


✅ Rated Star Wars: Episode V - The Empire Strikes Back (1980)!

🎉 Thank you! Your preferences have been saved.
